<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook08_Visualisations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matplotlib seaborn pandas pillow -q
print("Install complete.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')

from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image

# Make output dirs
Path("results/grids").mkdir(parents=True, exist_ok=True)
Path("results/charts").mkdir(parents=True, exist_ok=True)

In [ ]:
SEEDS=[42,1337,2026,7777,2213935]
target_prompts=json.loads(Path("prompts/target_prompts.json").read_text())
recovered_prompts=json.loads(Path("prompts/recovered_prompts.json").read_text())

dirs = {
    "Baseline":Path ("Outputs/baseline/target"),
    "Erased":Path ("Outputs/erased/target"),
    "Recovered":Path ("Outputs/recovered/target")
}

for prompt_idx in range(10):
  target=target_prompts[prompt_idx]
  recovered=recovered_prompts[prompt_idx]
  fig, XY = plt.subplots(3, 5, figsize=(20, 12))
  for row, (label, d) in enumerate(dirs.items()):
    for  col, seed in enumerate(SEEDS):
      path = d /f"p{prompt_idx:03d}_s{seed:06d}.png"
      if path.exists():
        XY[row, col].imshow(Image.open(path))
      XY[row,col].set_title(f"seed={seed}" if row==0 else "", fontsize=9)
      XY[row,col].axis("off")
    XY[row,0].set_ylabel(label, fontsize=14, rotation=90,labelpad=15)
    XY[row,0].axis("on")
    XY[row,0].set_xticks([]); XY[row, 0].set_yticks([])
    for spine in XY[row,0].spines.values():
      spine.set_visible(False)

    target_short = target if len(target) <= 90 else target[:87] + "..."
    recovered_short = recovered if len(recovered) <= 90 else recovered[:87] + "..."
    fig.suptitle(
        f"Target prompt {prompt_idx}: {target_short}\n"
        f"Recovered prompt: {recovered_short}",
        fontsize=11, y=1.0,
    )
    plt.tight_layout()
    plt.savefig(f"results/grids/target_p{prompt_idx:03d}.pdf", bbox_inches="tight")
    plt.savefig(f"results/grids/target_p{prompt_idx:03d}.png", bbox_inches="tight", dpi=120)
    plt.close()
    print(f"  Saved grid for prompt {prompt_idx}")

print(f"\n10 grids saved to results/grids/ (PDF + PNG)")

In [ ]:
# === Cell 3: Three-way target grids ===
# For each of the 10 target prompts, render a 3x5 grid:
#   row 1 = baseline, row 2 = erased, row 3 = recovered
#   columns = the 5 seeds
SEEDS = [42, 1337, 2026, 7777, 2213935]
target_prompts = json.loads(Path("prompts/target_prompts.json").read_text())
recovered_prompts = json.loads(Path("prompts/recovered_prompts.json").read_text())

dirs = {
    "Baseline":  Path("outputs/baseline/target"),
    "Erased":    Path("outputs/erased/target"),
    "Recovered": Path("outputs/recovered"),
}

for prompt_idx in range(10):
    target = target_prompts[prompt_idx]
    recovered = recovered_prompts[prompt_idx]

    fig, axes = plt.subplots(3, 5, figsize=(20, 12))

    for row, (label, d) in enumerate(dirs.items()):
        for col, seed in enumerate(SEEDS):
            path = d / f"p{prompt_idx:03d}_s{seed:06d}.png"
            if path.exists():
                axes[row, col].imshow(Image.open(path))
            axes[row, col].set_title(f"seed={seed}" if row == 0 else "", fontsize=9)
            axes[row, col].axis("off")
        # Row label on the left
        axes[row, 0].set_ylabel(label, fontsize=14, rotation=90, labelpad=15)
        axes[row, 0].axis("on")
        axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
        for spine in axes[row, 0].spines.values():
            spine.set_visible(False)

    # Title with both prompts
    target_short = target if len(target) <= 90 else target[:87] + "..."
    recovered_short = recovered if len(recovered) <= 90 else recovered[:87] + "..."
    fig.suptitle(
        f"Target prompt {prompt_idx}: {target_short}\n"
        f"Recovered prompt: {recovered_short}",
        fontsize=11, y=1.0,
    )
    plt.tight_layout()
    plt.savefig(f"results/grids/target_p{prompt_idx:03d}.pdf", bbox_inches="tight")
    plt.savefig(f"results/grids/target_p{prompt_idx:03d}.png", bbox_inches="tight", dpi=120)
    plt.close()
    print(f"  Saved grid for prompt {prompt_idx}")

print(f"\n10 grids saved to results/grids/ (PDF + PNG)")

In [ ]:
df = pd.read_csv("results/metrics.csv")
row = df.iloc[0]

fid_data = pd.DataFrame({
    "Metric": ["Fidelity\n(N=250)", "Efficacy\n(N=50)", "Resilience\n(N=50)"],
    "FID": [row["fidelity_FID"], row["efficacy_FID"], row["resilience_FID"]],
})
clip_data = pd.DataFrame({
    "Metric": [
        "Fidelity\nCLIP-T\nbaseline",
        "Fidelity\nCLIP-T\nerased",
        "Efficacy\nCLIP-I",
        "Resilience\nCLIP-I",
    ],
    "CLIP score": [
        row["fidelity_CLIP-T_baseline"],
        row["fidelity_CLIP-T_erased"],
        row["efficacy_CLIP-I"],
        row["resilience_CLIP-I"],
    ],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=fid_data, x="Metric", y="FID", ax=axes[0],
            hue="Metric", legend=False, palette="viridis")
axes[0].set_title("FID across COPYRIGHTMETER axes", fontsize=12)
axes[0].set_ylabel("FID (lower = closer to baseline)")

sns.barplot(data=clip_data, x="Metric", y="CLIP score", ax=axes[1],
            hue="Metric", legend=False, palette="rocket")
axes[1].set_title("CLIP-based metrics", fontsize=12)
axes[1].set_ylabel("Cosine similarity")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("results/charts/metrics_bar.pdf", bbox_inches="tight")
plt.savefig("results/charts/metrics_bar.png", bbox_inches="tight", dpi=120)
plt.show()

print("\nSaved metrics bar chart to results/charts/metrics_bar.{pdf,png}")

In [ ]:
n_grids = len(list(Path("results/grids").glob("*.pdf")))
n_charts = len(list(Path("results/charts").glob("*.pdf")))

print(f"Grids:  {n_grids} PDFs in results/grids/")
print(f"Charts: {n_charts} PDFs in results/charts/")

print("\nFinal artefact summary across the pipeline:")
print(f"  outputs/baseline/:  300 images")
print(f"  outputs/erased/:    300 images")
print(f"  outputs/recovered/:  50 images")
print(f"  checkpoints/esd_vangogh_*.safetensors")
print(f"  prompts/{{target,unrelated,recovered}}_prompts.json + vangogh_pairs.csv")
print(f"  logs/seeds_{{baseline,erased,recovered}}.json + training_log_*.txt")
print(f"  results/metrics.csv")
print(f"  results/grids/*.{{pdf,png}}")
print(f"  results/charts/metrics_bar.{{pdf,png}}")
print(f"\nPipeline complete.")